## Nota:
#### Este notebook é apenas um exemplo de como seria rodar localmente os modelos. A ideia é entender o código usado para treinar, avaliar e entender os modelos. Todos os experimentos deste notebook foram de fato executados (em um servidor). Os códigos utilizados para executar os experimentos no servidor estão disponíveis no diretório `server_scripts`, e as funções de treinamento e avaliação estão implementadas no diretório `src`

### Importando as funções e bibliotecas

In [1]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.data_processing import process_data
from src.multi_output_training import (train_ridge_regression, 
train_random_forest, train_lgbm, train_catboost, train_xgboost)
from src.regressor_training import (train_rf_regressor, 
train_lgbm_regressor, train_xgb_regressor)
from src.evaluation import evaluate_model_performance, save_results, save_model
from sklearn.model_selection import train_test_split
import pandas as pd

/home/leodemore/anaconda3/envs/ic/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Criando automaticamente um DataFrame processado e pronto para modelagem

In [ ]:
dataset_name = 'yeast'
input_path = f'../data/raw/{dataset_name}.csv'
output_path = f'../data/processed/proc_{dataset_name}.csv'

df = process_data(input_path=input_path, output_path=output_path)
df.head(3)

### Dividindo o dataset processado em treino e teste
Essa parte deve ser mantida aqui para ser reutilizável em modelos diferentes 

In [ ]:
# Essa parte é mantida no notebook para reutilização em todas os modelos
df = pd.read_csv('../data/processed/proc_birds.csv')

# Separar features e targets
X = df.drop(columns=['F1 (macro averaged by label)', 'Model Size', 'Model Size Log'])
y = df[['F1 (macro averaged by label)', 'Model Size Log']]

# Dividir em treino e teste 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Treinando modelos com MultiOutputRegressor
Estratégia de separar o problema de multitarget em targets únicos e treinar o modelo para prever cada um separadamente

#### Treinando o modelo Ridge Regression

In [ ]:
# Chamar a função que treina e otimiza o modelo
grid_search_results = train_ridge_regression(
    X_train=X_train, 
    y_train=y_train, 
    alpha_values=[5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0]
)

# O best_estimator_ é o modelo já treinado com o melhor hiperparâmetro 
model = grid_search_results.best_estimator_

# Fazer predições
predictions = model.predict(X_test)

# Obter e exibir as métricas 
final_metrics = evaluate_model_performance(y_test, predictions)
print("Relatório:")
for metric_name, score in final_metrics.items():
    print(f"{metric_name}: {score}")

# Salvar os resultados deste modelo
save_results(
    model_name='Ridge_MultiOutput',
    metrics=final_metrics,
    best_params=grid_search_results.best_params_,
    filepath='../reports/model_comparison_birds.csv'
)


# Salvar o modelo para que seja reutilizável
save_model(model, '../models/birds/ridge_multioutput.joblib')

#### Treinando o modelo RandomForest

In [ ]:
# Chamar a função que treina e otimiza o modelo
best_model, best_params, _ = train_random_forest(
    X_train=X_train, 
    y_train=y_train,
    n_trials=50  
)

# 'best_model' é o pipeline treinado com os melhores hiperparâmetros
predictions = best_model.predict(X_test)

# Obter e exibir as métricas 
final_metrics = evaluate_model_performance(y_test, predictions)

print("\nRelatório:")
for metric_name, score in final_metrics.items():
    print(f"{metric_name}: {score}")

# Salvar os resultados deste modelo
save_results(
    model_name='RandomForest_MultiOutput',
    metrics=final_metrics,
    best_params=best_params,
    filepath ='../reports/model_comparison_birds.csv'
)

# Salvar o modelo 
save_model(
    model=best_model, 
    filepath='../models/birds/random_forest_multioutput.joblib'
)

#### Treinando o modelo LightGBM

In [ ]:
# Chamar a função que treina e otimiza o modelo
best_model, best_params, _ = train_lgbm(
    X_train=X_train, 
    y_train=y_train, 
)

# Fazer predições
predictions = best_model.predict(X_test)

# Obter e exibir as métricas 
final_metrics = evaluate_model_performance(y_test, predictions)

print("Relatório:")
for metric_name, score in final_metrics.items():
    print(f"{metric_name}: {score}")

# Salvar os resultados deste modelo
save_results(
    model_name='LGBM_MultiOutput',
    metrics=final_metrics,
    best_params=best_params,
    filepath ='../reports/model_comparison_birds.csv'
)

# Salvar o modelo para que seja reutilizável
save_model(model, '../models/birds/lgbm_multioutput.joblib')

#### Treinando o modelo CatBoost

In [ ]:
# Chamar a função que treina e otimiza o modelo
best_model, best_params, _ = train_catboost(
    X_train=X_train, 
    y_train=y_train, 
)

# Fazer predições
predictions = best_model.predict(X_test)

# Obter e exibir as métricas 
final_metrics = evaluate_model_performance(y_test, predictions)

print("Relatório:")
for metric_name, score in final_metrics.items():
    print(f"{metric_name}: {score}")

# Salvar os resultados deste modelo
save_results(
    model_name='CatBoost_MultiOutput',
    metrics=final_metrics,
    best_params=best_params,
    filepath ='../reports/model_comparison_birds.csv'
)

# Salvar o modelo para que seja reutilizável
save_model(best_model, '../models/birds/catboost_multioutput.joblib')

#### Treinando o modelo XGBoost

In [ ]:
# Chamar a função que treina e otimiza o modelo
best_model, best_params, _ = train_xgboost(
    X_train=X_train, 
    y_train=y_train, 
)

# Fazer predições
predictions = best_model.predict(X_test)

# Obter e exibir as métricas 
final_metrics = evaluate_model_performance(y_test, predictions)

print("Relatório:")
for metric_name, score in final_metrics.items():
    print(f"{metric_name}: {score}")

# Salvar os resultados deste modelo
save_results(
    model_name='XGBoost_MultiOutput',
    metrics=final_metrics,
    best_params=best_params,
    filepath ='../reports/model_comparison_birds.csv'
)

# Salvar o modelo para que seja reutilizável
save_model(best_model, '../models/birds/xgb_multioutput.joblib')

### Treinando modelos com RegressorChain
Estratégia de treinar um modelo para prever uma target e usar essa saída para prever a outra target

#### Treinando o modelo Random Forest

In [ ]:
# Chamar a função que treina e otimiza o modelo
best_model, best_params, _ = train_rf_regressor(
    X_train=X_train, 
    y_train=y_train, 
    n_trials=50
)

# Fazer predições
predictions = best_model.predict(X_test)

# Obter e exibir as métricas 
final_metrics = evaluate_model_performance(y_test, predictions)

print("Relatório:")
for metric_name, score in final_metrics.items():
    print(f"{metric_name}: {score}")

# Salvar os resultados deste modelo
save_results(
    model_name='RandomForest_RegressorChain',
    metrics=final_metrics,
    best_params=best_params,
    filepath ='../reports/model_comparison_birds.csv'
)

# Salvar o modelo para que seja reutilizável
save_model(best_model, '../models/birds/rfregressor.joblib')

#### Treinando o modelo LightGBM

In [ ]:
# Chamar a função que treina e otimiza o modelo
best_model, best_params, _ = train_lgbm_regressor(
    X_train=X_train, 
    y_train=y_train, 
    n_trials=50
)

# Fazer predições
predictions = best_model.predict(X_test)

# Obter e exibir as métricas 
final_metrics = evaluate_model_performance(y_test, predictions)

print("Relatório:")
for metric_name, score in final_metrics.items():
    print(f"{metric_name}: {score}")

# Salvar os resultados deste modelo
save_results(
    model_name='LGBM_RegressorChain',
    metrics=final_metrics,
    best_params=best_params,
    filepath ='../reports/model_comparison_birds.csv'
)

# Salvar o modelo para que seja reutilizável
save_model(best_model, '../models/birds/lgbmregressor.joblib')

#### Treinando o modelo XGBoost

In [ ]:
# Chamar a função que treina e otimiza o modelo
best_model, best_params, _ = train_xgb_regressor(
    X_train=X_train, 
    y_train=y_train, 
    n_trials=50
)

# Fazer predições
predictions = best_model.predict(X_test)

# Obter e exibir as métricas 
final_metrics = evaluate_model_performance(y_test, predictions)

print("Relatório:")
for metric_name, score in final_metrics.items():
    print(f"{metric_name}: {score}")

# Salvar os resultados deste modelo
save_results(
    model_name='XGBoost_RegressorChain',
    metrics=final_metrics,
    best_params=best_params,
    filepath ='../reports/model_comparison_birds.csv'
)

# Salvar o modelo para que seja reutilizável
save_model(best_model, '../models/birds/xgbregressor.joblib')

### Treinando modelos com NativeAdaptation
Estratégia de treinar um modelo que leva em conta a correlação entre os targets de forma nativa

#### Treinando o modelo Random Forest

####

#### Treinando o modelo K-Neighbors Regressor

#### Treinando o modelo LightGBM

#### Treinando o modelo CatBoost

#### Treinando o modelo MLP Regressor

#### Treinando o modelo XGBoost Regressor